In [1]:
import os
import json
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List, Optional

from langchain_classic.prompts import load_prompt
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_community.vectorstores import FAISS

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

FAISS_PATH = "faiss_movies"
PROMPT_DIR = "prompts"

c:\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

print("벡터 수:", vectorstore.index.ntotal)

C:\Users\김소영\AppData\Local\Temp\ipykernel_1536\223185528.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


벡터 수: 9742


In [40]:
# retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [3]:
class QueryParseResult(BaseModel):
    intent: str
    genres: List[str] = Field(default_factory=list)
    min_year: Optional[int] = None
    max_year: Optional[int] = None
    min_rating: Optional[float] = None
    max_runtime: Optional[int] = None
    mood: List[str] = Field(default_factory=list)
    audience: List[str] = Field(default_factory=list)
    similar_to: List[str] = Field(default_factory=list)
    keywords: List[str] = Field(default_factory=list)


class PreferenceProfile(BaseModel):
    taste_summary: str
    preferred_genres: List[str] = Field(default_factory=list)
    preferred_moods: List[str] = Field(default_factory=list)
    preferred_themes: List[str] = Field(default_factory=list)

In [4]:
query_prompt = load_prompt(f"{PROMPT_DIR}/query_parsing.yaml", encoding="utf-8")
preference_prompt = load_prompt(f"{PROMPT_DIR}/preference_analysis.yaml", encoding="utf-8")
explanation_prompt = load_prompt(f"{PROMPT_DIR}/recommendation_explanation.yaml", encoding="utf-8")
followup_prompt = load_prompt(f"{PROMPT_DIR}/followup_update.yaml", encoding="utf-8")
final_response_prompt = load_prompt(f"{PROMPT_DIR}/final_response.yaml", encoding="utf-8")

C:\Users\김소영\AppData\Local\Temp\ipykernel_1536\4090854223.py:1: LangChainDeprecationWarning: The function `load_prompt` was deprecated in LangChain 1.2.21 and will be removed in 2.0.0. Use `Use `dumpd`/`dumps` from `langchain_core.load` to serialize prompts and `load`/`loads` to deserialize them.` instead.
  query_prompt = load_prompt(f"{PROMPT_DIR}/query_parsing.yaml", encoding="utf-8")


In [6]:
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

query_parser = PydanticOutputParser(pydantic_object=QueryParseResult)
preference_parser = PydanticOutputParser(pydantic_object=PreferenceProfile)
text_parser = StrOutputParser()

# structured prompt에 format_instructions 주입 

In [8]:
query_prompt = query_prompt.partial(
    format_instructions=query_parser.get_format_instructions()
)

preference_prompt = preference_prompt.partial(
    format_instructions=preference_parser.get_format_instructions()
)

followup_prompt = followup_prompt.partial(
    format_instructions=query_parser.get_format_instructions()
)

chain 구성

In [9]:
query_parsing_chain = query_prompt | llm | query_parser
preference_analysis_chain = preference_prompt | llm | preference_parser
recommendation_explanation_chain = explanation_prompt | llm | text_parser
followup_update_chain = followup_prompt | llm | query_parser
final_response_chain = final_response_prompt | llm | text_parser

retrieval 함수

In [10]:
def retrieve_movies(user_query, k=5):
    return vectorstore.similarity_search(user_query, k=k)

재정렬 함수

In [11]:
def rerank_results(results):
    scored = []
    for doc in results:
        avg_rating = doc.metadata.get("avg_rating", 0) or 0
        rating_count = doc.metadata.get("rating_count", 0) or 0
        score = float(avg_rating) + min(float(rating_count) / 1000, 1.0)
        scored.append((score, doc))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored]

Memory component 추가

In [12]:
from langchain_community.chat_message_histories import ChatMessageHistory
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

E2E 추천 함수

In [13]:
def build_recommendation_payload(user_query: str):
    # 1) 질의 구조화
    parsed_query = query_parsing_chain.invoke({
        "user_query": user_query
    })

    # 2) retrieval
    results = retrieve_movies(user_query, k=5)
    results = rerank_results(results)

    # 3) 각 영화별 추천 이유 생성
    movie_reasons = []
    for doc in results[:3]:
        reason = recommendation_explanation_chain.invoke({
            "user_query": user_query,
            "taste_profile": "",
            "movie_info": doc.page_content
        })
        movie_reasons.append({
            "title": doc.metadata.get("title"),
            "year": doc.metadata.get("year"),
            "reason": reason
        })

    return {
        "parsed_query": parsed_query,
        "movie_reasons": movie_reasons
    }

히스토리 반영 프롬프트

In [14]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

final_chat_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 CineMate라는 영화 추천 도우미입니다. "
     "과거 대화 히스토리를 참고하여 사용자의 취향과 직전 요청을 반영하세요. "
     "입력에 없는 영화 정보는 만들어내지 마세요. "
     "항상 한국어로 응답하세요."),
    MessagesPlaceholder(variable_name="history"),
    ("human",
     "사용자 요청:\n{user_query}\n\n"
     "추천 결과:\n{recommended_movies_with_reasons}")
])

Memory-aware 최종 응답 chain

In [15]:
memory_response_chain = final_chat_prompt | llm | text_parser

RunnableWithMessageHistory로 감싸기

In [16]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory

def invoke_cinemate_core(inputs: dict):
    user_query = inputs["user_query"]

    payload = build_recommendation_payload(user_query)

    return memory_response_chain.invoke({
        "history": inputs["history"],
        "user_query": user_query,
        "recommended_movies_with_reasons": json.dumps(
            payload["movie_reasons"],
            ensure_ascii=False,
            indent=2
        )
    })

cinemate_core = RunnableLambda(invoke_cinemate_core)

cinemate_with_memory = RunnableWithMessageHistory(
    cinemate_core,
    get_session_history,
    input_messages_key="user_query",
    history_messages_key="history",
)

End-to-End 실행

In [17]:
session_id = "user-001"

response1 = cinemate_with_memory.invoke(
    {"user_query": "가족이랑 보기 좋은 2시간 이하 감동적인 영화 추천해줘"},
    config={"configurable": {"session_id": session_id}}
)

print(response1)

가족과 함께 보기 좋은 2시간 이하의 감동적인 영화로는 2000년에 나온 "Joint Security Area"를 추천드립니다. 이 영화는 남북한 군인들의 인간적인 교류를 다루며 가족 간의 이해와 화합이라는 주제를 감동적으로 그려내어 가족과 함께 보기 좋은 작품입니다.


후속 질문

In [69]:
response2 = cinemate_with_memory.invoke(
    {"user_query": "조금 더 가볍고 웃긴 영화로 바꿔줘"},
    config={"configurable": {"session_id": session_id}}
)

print(response2)

가족과 함께 가볍고 웃기면서도 감동을 느낄 수 있는 영화로는 1985년에 나온 "Tampopo"를 추천드립니다. 이 영화는 음식과 삶을 유머러스하게 다루어 편안하고 즐거운 분위기 속에서 웃음과 감동을 함께 느낄 수 있어 가족 영화로 적합합니다.
